Logsitic Regression and Feature Scaling

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn import metrics
from sklearn.model_selection import train_test_split

I have one of three datasets that is designed to predict a binary yes or no, so that is the only dataset I will work with this week. The variable "DEP_DEL15" is 1 is a flight was delayed greater than 15 minutes at departure and 0 if it was not.

In [3]:
#Import 2nd Dataset, 2019 Flight Delay Info

delays_2019 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\2019_Delay_Data\full_data_flightdelay.csv")

delays_2019.head()

,MONTH,DAY_OF_WEEK,DEP_DEL15,DEP_TIME_BLK,DISTANCE_GROUP,SEGMENT_NUMBER,CONCURRENT_FLIGHTS,NUMBER_OF_SEATS,CARRIER_NAME,AIRPORT_FLIGHTS_MONTH,...,PLANE_AGE,DEPARTING_AIRPORT,LATITUDE,LONGITUDE,PREVIOUS_AIRPORT,PRCP,SNOW,SNWD,TMAX,AWND
0,1,7,0,0800-0859,2,1,25,143,Southwest Airlines Co.,13056,...,8,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
1,1,7,0,0700-0759,7,1,29,191,Delta Air Lines Inc.,13056,...,3,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
2,1,7,0,0600-0659,7,1,27,199,Delta Air Lines Inc.,13056,...,18,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
3,1,7,0,0600-0659,9,1,27,180,Delta Air Lines Inc.,13056,...,2,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
4,1,7,0,0001-0559,7,1,10,182,Spirit Air Lines,13056,...,1,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91


In [5]:
#The target variable for this dataset is "DEP_DEL15" it is a binary 1 if the flight was delayed more than 15 minutes in its departure delay else 0
delays_2019_mod = pd.get_dummies(delays_2019, columns=["DEP_TIME_BLK","DEPARTING_AIRPORT", "PREVIOUS_AIRPORT", "CARRIER_NAME"])

target_2019 = delays_2019_mod["DEP_DEL15"]
variables_2019 = delays_2019_mod.drop(["DEP_DEL15"], axis = 1)

In [6]:
#Train test split
X_train, X_test, y_train, y_test = train_test_split(variables_2019, target_2019, random_state=42)

I am going to use Precision as the scoring metric since this is predicting significantly delayed flights. Most flights are not delayed so I want to minimize false positives.

In [ ]:
#Optimize Random Forest Classification
rfc = RandomForestClassifier()
param_grid = {
    "max_depth" : [5,10,20,30,40],
    "min_samples_split" : [2,3,5,10,20,50],
    "min_samples_leaf" : [1,5,10,20,50],
    "ccp_alpha" : [0.0, 0.001, 0.005, 0.01, 0.05],
}

rfc_cv = RandomizedSearchCV(rfc, param_distributions=param_grid, n_iter=2, scoring = "precision").fit(X_train,y_train)

print(rfc_cv.best_params_)

C:\Users\lemrd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\lemrd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\lemrd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:18

Even simple parameter searches are taking longer than 180 minutes for me to run, so I am going to have to do this unoptimized.

In [7]:
rfc = RandomForestClassifier().fit(X_train,y_train)

train_pred = rfc.predict(X_train)
test_pred = rfc.predict(X_test)

train_prec = metrics.precision_score(y_train,train_pred)
test_prec = metrics.precision_score(y_test,test_pred)

print(f"The precision on the training data was {train_prec:.4f} and on the test data was {test_prec:.4f}")

The precision on the training data was 0.9996 and on the test data was 0.5976


This is clearly a case of overfitting. The model performed much worse on the test data, though still better than many of the previous models. I am only going to have time to run one more attempt at this, so I'm going to change some parameters to simplify the models.

In [9]:
rfc = RandomForestClassifier(max_depth=15,max_features=10,ccp_alpha=.01).fit(X_train,y_train)

train_pred = rfc.predict(X_train)
test_pred = rfc.predict(X_test)

train_prec = metrics.precision_score(y_train,train_pred)
test_prec = metrics.precision_score(y_test,test_pred)

print(f"The precision on the training data was {train_prec:.4f} and on the test data was {test_prec:.4f}")

The precision on the training data was 0.0000 and on the test data was 0.0000


C:\Users\lemrd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\lemrd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [2]:
#Import 2nd Dataset, 2019 Flight Delay Info
delays_2019 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\2019_Delay_Data\full_data_flightdelay.csv")

#The target variable for this dataset is "DEP_DEL15" it is a binary 1 if the flight was delayed more than 15 minutes in its departure delay else 0
delays_2019.drop(["DEP_TIME_BLK","DEPARTING_AIRPORT", "PREVIOUS_AIRPORT", "CARRIER_NAME"], axis=1, inplace=True)

target_2019 = delays_2019["DEP_DEL15"]
variables_2019 = delays_2019.drop(["DEP_DEL15"], axis = 1)

#Train test split
X_train, X_test, y_train, y_test = train_test_split(variables_2019, target_2019, random_state=42)

In [9]:
#Optimize Random Forest Classification
rfc = RandomForestClassifier()
param_grid = {
    "max_depth" : [15,20,50],
    "min_samples_split" : [2,3,5,10,20,50],
    "min_samples_leaf" : [1,5,10,20,50],
    "ccp_alpha" : [0.0, 0.001, 0.005, 0.01, 0.05],
}

rfc_cv = RandomizedSearchCV(rfc, param_distributions=param_grid, n_iter=5, scoring = "precision").fit(X_train,y_train)

print(rfc_cv.best_params_)

C:\Users\lemrd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\lemrd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\lemrd\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:18

{'min_samples_split': 2, 'min_samples_leaf': 50, 'max_depth': 20, 'ccp_alpha': 0.0}


In [3]:
rfc = RandomForestClassifier(min_samples_split=2,min_samples_leaf=50,max_depth=20).fit(X_train,y_train)

train_pred = rfc.predict(X_train)
test_pred = rfc.predict(X_test)

train_prec = metrics.precision_score(y_train,train_pred)
test_prec = metrics.precision_score(y_test,test_pred)

print(f"The precision on the training data was {train_prec:.4f} and on the test data was {test_prec:.4f}")

The precision on the training data was 0.7510 and on the test data was 0.7206
